# QNN — Iris Dataset


## Librerias


In [ ]:
# 1. Import all libraries
import pandas as pd
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.preprocessing import MinMaxScaler, label_binarize
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from qiskit.circuit.library import zz_feature_map, real_amplitudes
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.algorithms.classifiers import NeuralNetworkClassifier
from qiskit_machine_learning.optimizers import COBYLA

np.random.seed(42)


## Data


In [ ]:
data = pd.DataFrame(datasets.load_iris().data, columns=datasets.load_iris().feature_names)
data['target'] = datasets.load_iris().target
data


In [ ]:
data.target.unique()


## Data Preparation


In [ ]:
n_qubits = 4

X = data.drop(columns='target')
y = data['target']

# Split test set with a ratio of 0.2
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Custom Dataset Loaded: Train={len(X_train)}, Test={len(X_test)}")

# Normalize to [0, π]
minmax = MinMaxScaler(feature_range=(0, np.pi))
X_train = minmax.fit_transform(X_train)
X_test = minmax.transform(X_test)
X_test = np.clip(X_test, 0, np.pi)

print(f"Final shapes -> Train: {X_train.shape}, Test: {X_test.shape}")


## Quantum Neural Network


In [ ]:
# Feature map + ansatz (4 qubits, 3 salidas para las 3 clases)
feature_map = zz_feature_map(feature_dimension=n_qubits, reps=1)
ansatz = real_amplitudes(num_qubits=n_qubits, reps=1)
circuit = feature_map.compose(ansatz)

# Un observable Z por clase (qubits 0, 1 y 2)
observables = [
    SparsePauliOp("ZIII"),
    SparsePauliOp("IZII"),
    SparsePauliOp("IIZI"),
]

qnn = EstimatorQNN(
    circuit=circuit,
    estimator=StatevectorEstimator(),
    observables=observables,
    input_params=feature_map.parameters,
    weight_params=ansatz.parameters,
)

qnn_model = NeuralNetworkClassifier(
    neural_network=qnn,
    loss="squared_error",
    one_hot=True,
    optimizer=COBYLA(maxiter=100),
)


## Entrenamiento QNN


In [ ]:
# Training and timing statistics
print("=== Train QNN ===")
start = time.time()
qnn_model.fit(X_train, np.asarray(y_train))
qnn_time = time.time() - start
print(f"QNN training time: {qnn_time:.2f}s")


## Model Evaluation


In [ ]:
def evaluate(model, X_t, y_t, classes):
    preds = model.predict(X_t)
    scores = model.predict_proba(X_t)

    acc = accuracy_score(y_t, preds)
    prec = precision_score(y_t, preds, average="macro", zero_division=0)
    rec = recall_score(y_t, preds, average="macro", zero_division=0)
    f1 = f1_score(y_t, preds, average="macro", zero_division=0)
    auc = roc_auc_score(label_binarize(y_t, classes=classes), scores, average="macro")
    return acc, prec, rec, f1, auc, scores, preds


In [ ]:
classes = np.unique(y_train)
qnn_acc, qnn_prec, qnn_rec, qnn_f1, qnn_auc, qnn_probs, qnn_preds = evaluate(
    qnn_model, X_test, np.asarray(y_test), classes
)
print("\n=== Results ===")
print(
    f"QNN - Acc: {qnn_acc:.4f} | Prec: {qnn_prec:.4f} | Rec: {qnn_rec:.4f} | "
    f"F1: {qnn_f1:.4f} | AUC: {qnn_auc:.4f}"
)


In [ ]:
# Result Visualization (multiclase + paleta UNIR)
plt.rcParams.update({
    'font.sans-serif': ['Arial'],
    'axes.edgecolor': '#C0C0C8',
    'grid.color': '#C0C0C8',
    'grid.linestyle': '--'
})

def get_inference_time(model, data):
    start = time.time()
    _ = model.predict(data)
    return time.time() - start

qnn_inf_time = get_inference_time(qnn_model, X_test)

quantum_color = '#219BC3'  # unirazul
unir_gris = '#858591'

fig, axs = plt.subplots(1, 2, figsize=(14, 5), dpi=150)
fig.suptitle('QNN Performance Analysis', fontsize=18, y=1.02, color='#0E4A75')

metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC-ROC']
qnn_vals = [qnn_acc, qnn_prec, qnn_rec, qnn_f1, qnn_auc]
x = np.arange(len(metrics))
width = 0.5

for i in range(len(metrics)):
    axs[0].bar(x[i], qnn_vals[i], width, color=quantum_color,
               label='QNN' if i == 0 else '', zorder=3)
    axs[0].text(x[i], qnn_vals[i] + 0.02, f'{qnn_vals[i]:.4f}',
                ha='center', va='bottom', fontsize=9, color=unir_gris)

axs[0].set_ylabel('Score')
axs[0].set_title('Evaluation Metrics', color='#0E4A75')
axs[0].set_xticks(x)
axs[0].set_xticklabels(metrics)
axs[0].set_ylim(0, 1.35)
axs[0].legend(loc='upper right')
axs[0].grid(axis='y', zorder=0)

eff_labels = ['Training Time', 'Inference Time']
qnn_eff = [qnn_time, qnn_inf_time]
x_eff = np.arange(len(eff_labels))

axs[1].bar(x_eff, qnn_eff, width, color=quantum_color, label='QNN', zorder=3)
axs[1].set_xticks(x_eff)
axs[1].set_xticklabels(eff_labels)
axs[1].set_ylabel('Time (s)')
axs[1].set_title('Computational Efficiency', color='#0E4A75')
max_time = max(qnn_eff)
axs[1].set_ylim(0, max_time * 1.35 if max_time > 0 else 1)
axs[1].legend(loc='upper right')
axs[1].grid(axis='y', zorder=0)

offset = max_time * 0.02 if max_time > 0 else 0.02
for i in range(2):
    axs[1].text(x_eff[i], qnn_eff[i] + offset, f'{qnn_eff[i]:.4f}s',
                ha='center', va='bottom', fontsize=9, color=unir_gris)

plt.tight_layout()
plt.show()
